<h1>Approaches to Machine learning</h1>
<h2>Task 1</h2>

<p>"Mimicking the behavior of the Apple Reminders app when creating a "Groceries" reminder list, create a programme that when passed a grocery item name (eg. "oranges", "eggs") is able to return which category they belong in (eg. "Fruits and vegetable", "Eggs and Dairy", respectively).

You should start with collecting and processing necessary data that can contribute to completing this task. The focus of this task will be on this data collection and processing steps. What kind of data do you need, how will you process them, how much will you need, etc."
</p>

In [159]:
from nltk.tokenize import word_tokenize
import numpy as np
import pandas as pd
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import Perceptron
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

In [160]:
#loading in the dataset
df = pd.read_csv("hf://datasets/AmirMohseni/GroceryList/data.csv")

#filtering out produce items
prod_df = df[df["Category"].str.contains("Produce", case=False, na=False)]
print(prod_df)

#removing unwanted rows from the dataframe such as household items and pet toys
unwanted_index = []
i = 127
while i <= 170:
    unwanted_index.append(i)
    i+=1
df = df.drop(unwanted_index, axis=0)
df = df.reset_index(drop=True)

          Item Category
0        apple  Produce
1       banana  Produce
2       orange  Produce
3      lettuce  Produce
4       carrot  Produce
5     broccoli  Produce
6       tomato  Produce
7      spinach  Produce
8       grapes  Produce
9         pear  Produce
10        kiwi  Produce
11       mango  Produce
12  strawberry  Produce
13   blueberry  Produce
14    zucchini  Produce
15    cucumber  Produce
16     avocado  Produce


In [161]:
#adding new entries to even out each catagory
additionals = ["chicken thigh", "chicken breast", "pork belly", "kefir", "biscuits", "brown sauce", "chutney", "lasanga", "risotto", "basmati rice", "cider", "ale", "olives", "jamon", "lard", "canned butter beans", "canned jalpenos", "canned potatoes", "canned custard"]
#catagories of the new entries
additionals_cat = ["Meat & Seafood", "Meat & Seafood", "Meat & Seafood", "Dairy & Eggs", "Snacks", "Condiments & Sauces", "Condiments & Sauces", "Pasta & Grains", "Pasta & Grains", "Pasta & Grains", "Beverages", "Beverages", "Deli", "Deli", "Deli", "Canned", "Canned", "Canned", "Canned"]

#creating a dataframe from the new entries
additionals_df = pd.DataFrame({"Item" : additionals, "Category" : additionals_cat})
additionals_df

#adding the new entries to the original dataframe
df = pd.concat([df, additionals_df], ignore_index=True, axis =0)

In [162]:
#first i need to access the items catergory, secondly I create new data by taking the string and adding organic to the beginning
organic_data = []
columns_dict = ["Item", "Category"]
produce_list = "Produce"
for row in prod_df.iloc[:,0]:
    organic_data.append("organic " + row )

#Then making this list into a dict with Produce 
organic_dict = {value: produce_list for value in organic_data}
#organic_dict

#making the dict into a dataframe
organic_df = ( pd.DataFrame.from_dict(organic_dict, orient="index", columns=["Category"]).reset_index().rename(columns={"index": "Item"})
)

#adding it to the dataframe
df = pd.concat([df, organic_df], ignore_index=True, axis =0)
df

,Item,Category
0,apple,Produce
1,banana,Produce
2,orange,Produce
3,lettuce,Produce
4,carrot,Produce
...,...,...
212,organic strawberry,Produce
213,organic blueberry,Produce
214,organic zucchini,Produce
215,organic cucumber,Produce


In [163]:
#using the same logic as before, create a streamlined way to add descriptors to items in other catagories
descriptors = "ground","cultured","gluten free","low fat","low calorie","vegan","organic", "low soidum", "ready to eat", "boneless", "artisan", "garlic", "buckwheat", "low sugar", "smoked", "locally grown"

#for i in descriptors:
#    test_df = df["Item"].str.contains(i, case=False, na=False)
#print(test_df)    

class ItemExpander:
    def __init__(self, df, category, modifier):
        #initializing with dataframe, category to filter by, and modifier to add
        self.df = df
        self.category = category
        self.modifier = modifier

    def generate(self):
        #filtering the items by category
        cat_df = self.df[self.df["Category"].str.contains(self.category, case=False, na=False)]
        
        #creating modified items
        #skips if the item has more than one descriptor 
        new_items = [] 
        for item in cat_df["Item"]:
            counter = 0
            for i in descriptors:
                if i in item:
                    break
                if i not in item:
                    new_items.append(f"{self.modifier} {item}")
                    counter += 1
                if counter >= 1:
                    break
                

        #building new df
        new_df = pd.DataFrame({
            "Item": new_items,
            "Category": self.category
        })
        
        #appending it to the original df
        return pd.concat([self.df, new_df], ignore_index=True)

expander1 = ItemExpander(df, "Meat & Seafood", "ground")
df = expander1.generate()

expander2 = ItemExpander(df, "Dairy & Eggs", "cultured")
df = expander2.generate()

expander3 = ItemExpander(df, "Bakery", "gluten free")
df = expander3.generate()

expander4 = ItemExpander(df, "Meat & Seafood", "boneless")
df = expander4.generate()

expander5 = ItemExpander(df, "Snacks", "gluten free")
df = expander5.generate()

expander6 = ItemExpander(df, "Condiments & Sauces", "low fat")
df = expander6.generate()

expander7 = ItemExpander(df, "Pasta & Grains", "gluten free")
df = expander7.generate()

expander8 = ItemExpander(df, "Beverages", "low calorie")
df = expander8.generate()

expander9 = ItemExpander(df, "Deli", "vegan")
df = expander9.generate()

expander10 = ItemExpander(df, "Canned", "low sodium")
df = expander10.generate()

expander11 = ItemExpander(df, "Canned", "ready to eat")
df = expander11.generate()

expander12 = ItemExpander(df, "Dairy & Eggs", "organic")
df = expander12.generate()

expander13 = ItemExpander(df, "Bakery", "artisan")
df = expander13.generate()

expander14 = ItemExpander(df, "Snacks", "low calorie")
df = expander14.generate()

expander15 = ItemExpander(df, "Condiments & Sauces", "garlic")
df = expander15.generate()

expander16 = ItemExpander(df, "Pasta & Grains", "buckwheat")
df = expander16.generate()

expander17 = ItemExpander(df, "Beverages", "low sugar")
df = expander17.generate()

expander18 = ItemExpander(df, "Deli", "smoked")
df = expander18.generate()

expander19 = ItemExpander(df, "Produce", "locally grown")
df = expander19.generate()

df
#df.to_csv('extended_data10.txt', sep='\t', index=False)

,Item,Category
0,apple,Produce
1,banana,Produce
2,orange,Produce
3,lettuce,Produce
4,carrot,Produce
...,...,...
685,locally grown organic strawberry,Produce
686,locally grown organic blueberry,Produce
687,locally grown organic zucchini,Produce
688,locally grown organic cucumber,Produce


In [164]:
#using this to check how many produce items are in the dataframe
len(df[df["Category"].str.contains("Produce", case=False, na=False)])

68

In [165]:
#enquiring as to what product the user needs
enquiry = input(str("What food are you looking for? "))
#converts from nontype to string using the get() function, replacing "None" with an empty string (`''`) if "enquiry" is "None"
enquiry = str({None: ''}.get(enquiry, enquiry))

#tokenising the items and enquiry
tokens = [word_tokenize(item) for item in df["Item"]]
token_enquiry = word_tokenize(enquiry)

#initialize an empty set for the vocabulary
vocabulary = set()

#build the vocabulary
for sentence in tokens:
    vocabulary.update(sentence)

#convert to a sorted list
vocabulary = sorted(list(vocabulary))

def vector_create(sentence, vocab):
    #initialize a vector of zeros
    vector = [0] * len(vocab)  
    for word in sentence:
        if word in vocab:
            #find the index of the word in the vocabulary
            idx = vocab.index(word)  
             #increment the count at that index
            vector[idx] += 1 
    return vector

x = [vector_create(sentence, vocabulary) for sentence in tokens]
x_test = vector_create(token_enquiry, vocabulary)
y = df["Category"].tolist()
#making the data 2D for ease later on
x_test_2d = [x_test]
#for vector in bag_vectors_enquiry:

In [ ]:
y = pd.Series.to_list(df["Category"])

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size = 0.1, random_state = 13)

clf = OneVsRestClassifier(Perceptron(class_weight='balanced', max_iter=1000, random_state=0, tol=1e-3))
clf.fit(x, y)
y_pred = clf.predict(x_test)

print(enquiry)

# Evaluating the model
print(accuracy_score(y_test, y_pred)*100)
print(classification_report(y_test, y_pred))


apple
97.10144927536231
                     precision    recall  f1-score   support

             Bakery       1.00      1.00      1.00         9
          Beverages       1.00      1.00      1.00        10
             Canned       1.00      1.00      1.00         4
       Canned Goods       1.00      1.00      1.00         2
Condiments & Sauces       1.00      0.83      0.91         6
       Dairy & Eggs       1.00      1.00      1.00         6
               Deli       1.00      1.00      1.00         6
       Frozen Foods       1.00      1.00      1.00         2
     Meat & Seafood       1.00      0.80      0.89         5
             Pantry       0.00      0.00      0.00         0
     Pasta & Grains       1.00      1.00      1.00         5
            Produce       1.00      1.00      1.00         8
             Snacks       1.00      1.00      1.00         6

           accuracy                           0.97        69
          macro avg       0.92      0.89      0.91        6

c:\Users\Elune\miniconda3\envs\big-data-venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Elune\miniconda3\envs\big-data-venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Elune\miniconda3\envs\big-data-venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
#get raw scores
scores = clf.decision_function(x_test_2d)  

#find top two classes
best_idx = np.argmax(scores[0])
#second highest
second_best = np.partition(scores[0], -2)[-2]  

#setting margin for confidence
margin = 0.5 
#making prediction based on margin
if (scores[0][best_idx] - second_best) >= margin:
    prediction = clf.classes_[best_idx]
else:
    #if the margin is under 0.5, categorise as uncategorised
    prediction = "Uncategorised"

print(f"Input: {enquiry} → Category: {prediction}")
print(f"Top Score: {scores[0][best_idx]:.2f}, Second Score: {second_best:.2f}")

Input: apple → Predicted Category: Produce
Top Score: 2.58, Second Score: -2.36
